[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mahsa-agz/mahsa-agz.github.io/blob/main/ds-handbook/exam_prep/day_09/3_stats.ipynb)

# Day 9: Statistics, A/B testing and product (easy)

**How to work:** Answer in two layers: the number (code) and one plain sentence a product manager understands. Name the method and its assumptions before computing.

**Today (25 min):**

| # | Question | Level | Minutes |
|---|---|---|---|
| Q1 | Pick the test, fast | easy | 6 |
| Q2 | Class and survival: one test for a whole table | easy | 6 |
| Q3 | Rounds played: which test? | easy | 8 |
| Q4 | The landing page test (review) | easy | 5 |

**Handbook:** [Statistics, A/B testing and product](https://mahsa-agz.github.io/ds-handbook/#stats), [cheat sheets](https://mahsa-agz.github.io/ds-handbook/#cheat)

## Data

All real datasets (see DATASETS.md for sources). The setup cell loads them:

- `titanic`: 891 passengers (`survived` 0/1, `pclass`, `sex`, `age`, `fare`, `class`, `who` ...).
- `cookie`: Cookie Cats mobile game A/B test, 90,189 players. `version` is `gate_30` (control) or `gate_40` (treatment: the first gate moved from level 30 to 40); `sum_gamerounds` = rounds played in the first week; `retention_1`, `retention_7` = came back 1 / 7 days after install (bool).
- `ab`: Udacity landing-page A/B test, cleaned in the setup to one row per user (`user_id`, `group` control/treatment, `converted` 0/1). `ab_raw` is the raw file.

Classic probability questions need no data: solve them on paper first, then check with a simulation (`rng = np.random.default_rng(0)`).

## Setup

Run this cell once before the questions.

In [ ]:
# Finds the data folder (local copy of the repository, or /content/data in Colab) and downloads a missing file
# from the ds-handbook repository on GitHub (or from its original source).
import os, sys, urllib.request

RAW = "https://raw.githubusercontent.com/mahsa-agz/mahsa-agz.github.io/main/ds-handbook/exam_prep/data/"

def _find_data():
    if "google.colab" in sys.modules:
        os.makedirs("/content/data", exist_ok=True)
        return "/content/data"
    for p in ("../data", "data", "exam_prep/data"):
        if os.path.isdir(p):
            return p
    os.makedirs("data", exist_ok=True)
    return "data"

DATA = _find_data()
print("data folder:", DATA)

def data_path(name, url=None):
    """Local path of a data file; downloads it from the repository (or from url) if it is not there yet."""
    path = os.path.join(DATA, name)
    if not os.path.exists(path):
        for source in (RAW + name, url):
            if not source:
                continue
            try:
                print("downloading", name)
                urllib.request.urlretrieve(source, path)
                break
            except Exception as e:
                print("  could not download from", source, "->", e)
    return path

import numpy as np
import pandas as pd
from scipy import stats as st
pd.set_option("display.width", 120)

titanic = pd.read_csv(data_path("titanic.csv", "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/titanic.csv"))
cookie = pd.read_csv(data_path("cookie_cats.csv", "https://raw.githubusercontent.com/robguilarr/ab_testing_cookie_cats/main/datasets/cookie_cats.csv"))
ab_raw = pd.read_csv(data_path("ab_data.csv", "https://raw.githubusercontent.com/F-Zarian/Analyze_AB_test_results/main/ab_data.csv"))
# clean copy: keep rows where group and page agree, one row per user (the raw file has both problems)
ab = ab_raw[(ab_raw["group"] == "treatment") == (ab_raw["landing_page"] == "new_page")]
ab = ab.drop_duplicates("user_id").reset_index(drop=True)
for _name in ['titanic', 'cookie', 'ab']:
    print(_name, globals()[_name].shape)

---
### Q1. Pick the test, fast

*easy, about 6 min*

Rapid fire, as in a phone screen. For each scenario name the test (and one assumption or caveat) in one line:

1. Checkout conversion, control versus new button, 80,000 users per arm.
2. Average order value per user (heavy right tail), control versus treatment, 50,000 users per arm.
3. The same 40 users do a task on the old and the new design; we measure seconds to finish.
4. Click-through rate for 4 different banner colours.
5. Mean session length in 3 regions.
6. A pilot: 12 of 40 users converted; the historical rate is 20%.
7. Is app rating related to the number of downloads?

**Then say it to a PM:** one or two plain sentences with the number and what it means for the product (no jargon).

*Your answer (double-click to write):*

<details><summary><b>Hint 1</b></summary>

Signal: for every scenario ask three questions: what type is the metric (0/1, numeric, counts)? How many groups? Are the groups independent or paired? Topic: the test chooser.

</details>

<details><summary><b>Hint 2</b></summary>

0/1 metric with 2 groups -> z-test for proportions (or chi-square). Numeric, 2 independent groups -> Welch t-test (rank test or bootstrap if small and skewed). Same units twice -> paired test. More than 2 groups -> chi-square (0/1) or ANOVA (numeric). Small counts -> exact test. Two numeric variables -> correlation.

</details>

<details><summary><b>Solution</b></summary>

**Model answer.**

| # | Test | Assumption or caveat |
|---|---|---|
| 1 | Two-proportion z-test (same as a 2x2 chi-square test) | Independent users; large counts (at least about 10 successes and 10 failures per arm). |
| 2 | Welch t-test on the means; check with a bootstrap CI | With 50,000 per arm the CLT makes the mean normal, but extreme values inflate the variance: cap (winsorize) or use CUPED for power. Mann-Whitney tests a shift in the *distribution*, not the mean, so it answers a different question. |
| 3 | Paired t-test on the 40 differences (or Wilcoxon signed-rank if the differences are skewed) | Pairing removes person-to-person variation; an unpaired test would waste it. Randomise the order of old and new to avoid a learning effect. |
| 4 | Chi-square test of independence on the 4 x 2 table (clicked / not clicked) | It only says 'some colour differs'; follow up with pairwise tests and a multiple-testing correction. |
| 5 | One-way ANOVA (Welch ANOVA if the spreads differ), or Kruskal-Wallis for skewed data | Same issue: a significant result needs post-hoc pairwise comparisons (Tukey). Regions are not randomised, so differences are not causal. |
| 6 | Exact binomial test (`st.binomtest(12, 40, 0.2)`) | n is small, so avoid the normal approximation. The pilot users may not be representative. |
| 7 | Spearman rank correlation (downloads are extremely skewed), or Pearson on log(downloads) | Correlation is not causation: popular apps get more ratings and more visibility. |

**Say it to a PM:** For each question, the test is just the tool; what you would tell a PM is the size of the difference with an error bar and whether it is big enough to act on, for example: 'the new button lifts conversion by 0.5 points, give or take 0.2'.

**Why:** Examiners rarely want formulas here; they want the decision path: metric type, number of groups, independent or paired, sample size and shape. Saying the caveat (what the test does *not* tell you) is what separates a strong answer from a memorised list.

**Common mistakes:** Using a t-test for a 0/1 metric in a small sample, or for paired data. Running 6 pairwise t-tests for 4 groups without correction. Saying Mann-Whitney compares medians (it compares whole distributions; it equals a median test only under extra assumptions).

**Learn more:** [stats-choosing-a-test](https://mahsa-agz.github.io/ds-handbook/#stats-choosing-a-test), [cheat-stats-tests](https://mahsa-agz.github.io/ds-handbook/#cheat-stats-tests)

</details>

---
### Q2. Class and survival: one test for a whole table

*easy, about 6 min*

Using `titanic`, is survival independent of passenger class (`pclass`, 3 levels)?
1. Build the 3 x 2 contingency table with `pd.crosstab`.
2. Run a chi-square test of independence and print the statistic, degrees of freedom and p-value.
3. Print the table of expected counts under H0, and the survival rate per class.
4. Why not run three separate two-proportion tests instead?

**Then say it to a PM:** one or two plain sentences with the number and what it means for the product (no jargon).

In [ ]:
# titanic is loaded in the setup

<details><summary><b>Hint 1</b></summary>

Signal: two categorical variables, one of them with more than 2 levels. Topic: the chi-square test of independence.

</details>

<details><summary><b>Hint 2</b></summary>

1. `tab = pd.crosstab(titanic.pclass, titanic.survived)`. 2. `chi2, p, dof, expected = st.chi2_contingency(tab)`. 3. `dof = (rows - 1) (cols - 1) = 2`. 4. Check all expected counts are at least 5.

</details>

<details><summary><b>Solution</b></summary>

```python
tab = pd.crosstab(titanic["pclass"], titanic["survived"])
chi2, p, dof, expected = st.chi2_contingency(tab)
print(tab.to_string())
print(f"chi2 {chi2:.1f}   dof {dof}   p {p:.1e}")
print("expected counts under H0:")
print(pd.DataFrame(expected, index=tab.index, columns=tab.columns).round(1).to_string())
print("survival rate by class:", titanic.groupby("pclass")["survived"].mean().round(3).to_dict())

# Output:
# survived    0    1
# pclass
# 1          80  136
# 2          97   87
# 3         372  119
# chi2 102.9   dof 2   p 4.5e-23
# expected counts under H0:
# survived      0      1
# pclass
# 1         133.1   82.9
# 2         113.4   70.6
# 3         302.5  188.5
# survival rate by class: {1: 0.63, 2: 0.473, 3: 0.242}
```

**Why:** The chi-square test compares observed counts with the counts expected if class and survival were independent (`row total x column total / grand total`). One test with 2 degrees of freedom answers 'does survival differ across classes at all' without multiplying the false-positive risk; the pairwise tests come after, with a correction. All expected counts are far above 5, so the chi-square approximation is valid (otherwise use Fisher's exact test). The p-value is tiny, but remember that the test says nothing about *why*: class is mixed up with sex and cabin position.

**Say it to a PM:** Survival clearly depended on ticket class: 63% in first class, 47% in second and 24% in third. A gap this large could not come from chance.

**Common mistakes:** Running the test on percentages instead of counts. Forgetting that a significant chi-square does not say which class differs. Using chi-square when some expected counts are below 5.

**Learn more:** [stats-choosing-a-test](https://mahsa-agz.github.io/ds-handbook/#stats-choosing-a-test), [stats-hypothesis-tests](https://mahsa-agz.github.io/ds-handbook/#stats-hypothesis-tests)

</details>

---
### Q3. Rounds played: which test?

*easy, about 8 min*

In `cookie`, compare `sum_gamerounds` between `gate_30` and `gate_40` (remove the one 49,854-round account first).
1. Print the mean and median per version.
2. Run a Welch t-test and a Mann-Whitney U test and print both p-values.
3. Bootstrap a 95% CI for the difference in **means** (`gate_40 - gate_30`, 1,000 resamples, `rng = np.random.default_rng(0)`).
4. The two tests disagree a little. Explain what each one tests and which one answers the business question 'did players play more rounds?'.

**Then say it to a PM:** one or two plain sentences with the number and what it means for the product (no jargon).

In [ ]:
# cookie is loaded in the setup
c = cookie[cookie.sum_gamerounds < 49854]

<details><summary><b>Hint 1</b></summary>

Signal: a heavy-tailed engagement metric with large groups. Topic: choosing between a test on means (Welch t, bootstrap) and a rank test (Mann-Whitney), and knowing they ask different questions.

</details>

<details><summary><b>Hint 2</b></summary>

1. `a, b = c[c.version == 'gate_30'].sum_gamerounds, ...`. 2. `st.ttest_ind(b, a, equal_var=False)`, `st.mannwhitneyu(b, a, alternative='two-sided')`. 3. Resample each group separately with `rng.choice(x, size=(1000, len(x)))`, take row means, subtract, use percentiles 2.5 and 97.5.

</details>

<details><summary><b>Solution</b></summary>

```python
c = cookie[cookie["sum_gamerounds"] < 49854]
a = c.loc[c["version"] == "gate_30", "sum_gamerounds"].to_numpy()
b = c.loc[c["version"] == "gate_40", "sum_gamerounds"].to_numpy()
print(f"gate_30 mean {a.mean():.2f} median {np.median(a):.0f}   gate_40 mean {b.mean():.2f} median {np.median(b):.0f}")
print(f"Welch t-test p {st.ttest_ind(b, a, equal_var=False).pvalue:.3f}")
print(f"Mann-Whitney p {st.mannwhitneyu(b, a, alternative='two-sided').pvalue:.3f}")
rng = np.random.default_rng(0)
diffs = rng.choice(b, size=(1000, len(b))).mean(axis=1) - rng.choice(a, size=(1000, len(a))).mean(axis=1)
lo, hi = np.percentile(diffs, [2.5, 97.5])
print(f"bootstrap 95% CI for the mean difference: [{lo:.2f}, {hi:.2f}] rounds")

# Output:
# gate_30 mean 51.34 median 17   gate_40 mean 51.30 median 16
# Welch t-test p 0.949
# Mann-Whitney p 0.051
# bootstrap 95% CI for the mean difference: [-1.30, 1.31] rounds
```

**Why:** Welch's t-test and the bootstrap ask: 'is the *mean* number of rounds different?'. That is the business question when total engagement matters (total rounds = mean x players). The mean difference is tiny (0.04 rounds) and its CI, about -1.3 to +1.3 rounds, is centred on 0, so we cannot claim a change in rounds played. Mann-Whitney asks a different question: 'does a random gate_40 player tend to rank above a random gate_30 player?'. It is robust to outliers and sensitive to shifts among light players, which is why its p-value differs. Pick the test from the question and decide it before looking at the data; running both and reporting the smaller p-value is p-hacking.

**Say it to a PM:** Players in the two versions played about the same number of rounds in their first week: the difference is under one round per player and could be chance. The real effect of the change is on retention, not on how much people play.

**Common mistakes:** Saying 'the data is not normal, so use Mann-Whitney' with 45,000 users per arm (the CLT covers the mean). Saying Mann-Whitney compares medians. Reporting whichever test is significant.

**Learn more:** [stats-choosing-a-test](https://mahsa-agz.github.io/ds-handbook/#stats-choosing-a-test), [stats-confidence-intervals](https://mahsa-agz.github.io/ds-handbook/#stats-confidence-intervals)

</details>

---
### Q4. The landing page test (review)

*easy, about 5 min*

Review of hypothesis tests. Using the cleaned `ab` table (one row per user):
1. Print the conversion rate and n for control and treatment.
2. Run a two-sided two-proportion z-test (`proportions_ztest`) and print z and p.
3. The company only cares whether the new page is **better**. Print the one-sided p-value for H1: treatment > control. What do you conclude?

**Then say it to a PM:** one or two plain sentences with the number and what it means for the product (no jargon).

In [ ]:
# ab is loaded and cleaned in the setup
from statsmodels.stats.proportion import proportions_ztest

<details><summary><b>Hint 1</b></summary>

Signal: two independent groups, 0/1 metric, large n. Topic: the two-proportion z-test, and the direction of a one-sided alternative.

</details>

<details><summary><b>Hint 2</b></summary>

1. `g = ab.groupby('group').converted.agg(['sum', 'count'])`. 2. `proportions_ztest([k_t, k_c], [n_t, n_c])`. 3. `alternative='larger'` tests treatment > control when treatment comes first.

</details>

<details><summary><b>Solution</b></summary>

```python
from statsmodels.stats.proportion import proportions_ztest
g = ab.groupby("group")["converted"].agg(["sum", "count", "mean"])
print("per group:")
print(g.round(4).to_string())
k = [g.loc["treatment", "sum"], g.loc["control", "sum"]]
n = [g.loc["treatment", "count"], g.loc["control", "count"]]
z, p2 = proportions_ztest(k, n)
_, p1 = proportions_ztest(k, n, alternative="larger")
print(f"z {z:.3f}   two-sided p {p2:.3f}   one-sided p (treatment > control) {p1:.3f}")

# Output:
# per group:
#              sum   count    mean
# group
# control    17489  145274  0.1204
# treatment  17264  145310  0.1188
# z -1.311   two-sided p 0.190   one-sided p (treatment > control) 0.905
```

**Why:** The treatment converts slightly *worse* (11.88% versus 12.04%), so z is negative. The two-sided p-value is 0.19: the gap is compatible with chance. For the one-sided alternative 'treatment is better', the observed result points the wrong way, so the p-value is large (0.90): there is no evidence at all that the new page is better. A one-sided test is fine only if the direction is chosen before the data, and it never turns a negative result into a positive one.

**Say it to a PM:** The new page does not convert better: it is 0.16 points lower, and the gap is small enough to be chance. With about 145,000 users per group the test could detect a lift of about a third of a point, so we can keep the old page.

**Common mistakes:** Passing the groups in the wrong order with `alternative='larger'` (that tests control > treatment). Switching to one-sided after seeing the result. Analysing the raw file with duplicate users and mismatched pages.

**Learn more:** [stats-hypothesis-tests](https://mahsa-agz.github.io/ds-handbook/#stats-hypothesis-tests), [stats-ab-analysis](https://mahsa-agz.github.io/ds-handbook/#stats-ab-analysis)

</details>

---
## Redo list

Write down every question you got wrong, solved only after Hint 2, or finished over time. Solve it again from scratch 3 days and 7 days later.

**Next:** [AI and ML](https://colab.research.google.com/github/mahsa-agz/mahsa-agz.github.io/blob/main/ds-handbook/exam_prep/day_09/4_ai.ipynb)